# Music Store Support Agents — walkthrough

This notebook is a guided tour of the production code in `src/`. It is not a
second copy of the implementation: every cell imports the same modules the CLI
uses, so what you see here is what `python -m src.run` runs.

**Run it top to bottom from the repository root.** You need the dependencies
installed (`pip install -r requirements-dev.txt`) and `GROQ_API_KEY` set in
`.env` for sections 4–6. Sections 1–3 run without an API key.

## 0. Environment

Configuration is read once, from the environment, by `src/config.py`. Nothing
else in the code base touches `os.environ`.

In [ ]:
from src.config import get_settings

settings = get_settings()
print("model:            ", settings.model_name)
print("LLM key present:  ", settings.has_llm_credentials)
print("Chinook source:   ", settings.chinook_sql_url)

## 1. The database

Chinook is downloaded as a SQL script and executed into an in-memory SQLite
connection. `StaticPool` keeps all sessions on that one connection, so every
tool sees the same data without any database server.

In [ ]:
from src.db import get_table_names

get_table_names()

In [ ]:
from sqlalchemy import text

from src.db import engine

with engine.connect() as conn:
    rows = conn.execute(
        text(
            """
            SELECT c.CustomerId, c.FirstName || ' ' || c.LastName AS Customer,
                   c.Phone, c.Email, COUNT(i.InvoiceId) AS Invoices
            FROM Customer c
            LEFT JOIN Invoice i ON i.CustomerId = c.CustomerId
            WHERE c.CustomerId IN (1, 2)
            GROUP BY c.CustomerId
            """
        )
    ).fetchall()

for row in rows:
    print(dict(row._mapping))

## 2. Tools

The agents only ever see these functions. Each is a LangChain `@tool` with a
typed signature, so the model gets a JSON schema for its arguments, and each
query uses bound parameters.

Note the `_require_nonblank` guard: if the model calls a tool without a
required argument, the tool raises an `interrupt()` and waits for the customer
instead of guessing.

In [ ]:
from src.tools.music_tools import get_albums_by_artist, get_songs_by_genre

for album in get_albums_by_artist.invoke({"artist": "Rolling Stones"}):
    print(f"{album['Title']}  (AlbumId={album['AlbumId']})")

print()
rock = get_songs_by_genre.invoke({"genre": "Rock"})
print(f"get_songs_by_genre('Rock') -> {len(rock)} rows, first three:")
for song in rock[:3]:
    print(" ", song["ArtistName"], "—", song["TrackName"])

In [ ]:
from src.tools.invoice_tools import (
    get_employee_by_invoice_and_customer,
    get_invoices_by_customer_sorted_by_date,
)

invoices = get_invoices_by_customer_sorted_by_date.invoke({"customer_id": "1"})
latest = invoices[0]

print("Most recent invoice:", latest["InvoiceDate"], f"${latest['Total']}")
print(
    "Support rep:",
    get_employee_by_invoice_and_customer.invoke(
        {"invoice_id": str(latest["InvoiceId"]), "customer_id": "1"}
    )["FirstName"],
)
print()
print("A different customer cannot read that invoice:")
print(
    get_employee_by_invoice_and_customer.invoke(
        {"invoice_id": str(latest["InvoiceId"]), "customer_id": "2"}
    )
)

## 3. The graph

Seven nodes, compiled once per process by `src/graph.py`. Building the graph
does not create an LLM client, which is why the offline tests can import it.

The diagram below is generated from the compiled graph itself.

In [ ]:
from src.graph import get_graph

graph, store = get_graph()

print(sorted(graph.get_graph().nodes))

In [ ]:
print(graph.get_graph().draw_mermaid())

## 4. Scenario 1 — verified customer, combined request

The message contains a phone number **and** asks about both a purchase and the
catalog, so the expected path is:

`verify_customer → load_memory → supervisor(both) → music_catalog →
invoice_info → synthesize → save_memory`

Interrupts, if any, are resolved in the loop below; with a valid phone number
there should be none.

In [ ]:
import logging

from langchain_core.messages import AIMessage, HumanMessage
from langgraph.types import Command

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

thread_1 = {"configurable": {"thread_id": "notebook-scenario-1"}}

result = graph.invoke(
    {
        "messages": [
            HumanMessage(
                content=(
                    "My phone number is +55 (12) 3923-5555. How much was my most "
                    "recent purchase? What albums do you have by the Rolling Stones?"
                )
            )
        ]
    },
    config=thread_1,
)

while "__interrupt__" in result:
    print(result["__interrupt__"][0].value["question"])
    result = graph.invoke(Command(resume=input("You: ")), config=thread_1)

print("verified customer_id:", result["customer_id"])
for message in result["messages"]:
    if (
        isinstance(message, AIMessage)
        and isinstance(message.content, str)
        and message.content.strip()
    ):
        print(message.content)

### Follow-up: long-term memory

`save_memory` extracted the artist and genre from the turn above. The next
message does not name an artist — the specialists only know about it because
`load_memory` put the summary into `loaded_memory` at the start of the turn.

In [ ]:
item = store.get(("customer", "1", "preferences"), "summary")
print("stored preferences:", item.value if item else "(none yet)")

before = len(result["messages"])
result = graph.invoke(
    {"messages": [HumanMessage(content="List some songs that match my preferences?")]},
    config=thread_1,
)

for message in result["messages"][before:]:
    if (
        isinstance(message, AIMessage)
        and isinstance(message.content, str)
        and message.content.strip()
    ):
        print(message.content)

## 5. Scenario 2 — no credentials in a new session

A new `thread_id` means no verified `customer_id`. `verify_customer` cannot
resolve an identity, so the graph stops at an interrupt instead of answering.

In [ ]:
thread_2 = {"configurable": {"thread_id": "notebook-scenario-2"}}

paused = graph.invoke(
    {
        "messages": [
            HumanMessage(
                content=(
                    "How much was my most recent purchase? What albums do you have "
                    "by the Rolling Stones?"
                )
            )
        ]
    },
    config=thread_2,
)

assert "__interrupt__" in paused, "expected the graph to ask for credentials"
print("paused:", paused["__interrupt__"][0].value)

Resuming with a valid identity continues from the interrupted node — the
original question is still in the checkpointed message history, so it does not
need to be repeated.

In [ ]:
resumed = graph.invoke(Command(resume="My customer id is 1"), config=thread_2)

print("verified customer_id:", resumed["customer_id"])
for message in resumed["messages"]:
    if (
        isinstance(message, AIMessage)
        and isinstance(message.content, str)
        and message.content.strip()
    ):
        print(message.content)

## 6. Where to go next

- **CLI:** `python -m src.run --demo` runs both scenarios without a notebook.
- **Tests:** `pytest -m "not e2e"` covers tools, verification, memory, HITL and
  graph wiring with no API key; `pytest` adds the live scenarios.
- **Design notes:** [`docs/architecture.md`](docs/architecture.md) explains the
  state contract, interrupt mechanics and failure behaviour.

Both `InMemorySaver` and `InMemoryStore` are process-local: restart the kernel
and sessions and preferences are gone. `build_graph()` is the single place to
swap them for persistent implementations.